# ⚖️ Actividad 04: Normalización y Escalado Algorítmico
---
**Módulo 2: Feature Engineering para LSTM**

Las redes LSTM usan funciones de activación `tanh` (salida entre −1 y 1) y `sigmoid` (entre 0 y 1). Si la red recibe variables con rangos muy distintos —como toneladas de producción vs. temperaturas en °C— los gradientes se distorsionan y el entrenamiento se vuelve inestable. El escalado garantiza que **todas las variables contribuyan en igualdad de condiciones** al aprendizaje del modelo.

## 🎯 Objetivos
1. **Identificar** variables numéricas continuas a escalar (excluyendo fechas, IDs y columnas ya acotadas cíclicamente entre −1 y 1).
2. **Ajustar `MinMaxScaler`** únicamente sobre el **conjunto de entrenamiento** (primeros 80% cronológicos) para evitar *data leakage*.
3. **Transformar** el dataset completo con ese mismo scaler.
4. **Serializar los scalers** en `notebooks/fase2/scalers/` con `joblib` para reutilizarlos en la fase de inferencia futura.
5. **Exportar** el dataset final, listo para ser consumido por la LSTM.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler

# Configuración estética
sns.set_theme(style='whitegrid', palette='viridis')
%matplotlib inline

# ── Rutas ──────────────────────────────────────────────────────────────────────
INPUT_FILE   = '../../data/processed/master_dataset_fase2_lags.csv'
OUTPUT_FILE  = '../../data/processed/master_dataset_fase2_final.csv'
SCALERS_DIR  = 'scalers/'

os.makedirs(SCALERS_DIR, exist_ok=True)
print(f"Cargando dataset desde: {INPUT_FILE}")
print(f"Scalers se guardarán en: {os.path.abspath(SCALERS_DIR)}")


## 1. Carga del Dataset con Rezagos

In [ ]:
df = pd.read_csv(INPUT_FILE)
df['fecha_evento'] = pd.to_datetime(df['fecha_evento'])

print(f"Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]} columnas")
print(f"Rango temporal: {df['fecha_evento'].min().date()} → {df['fecha_evento'].max().date()}")
df.head(2)


## 2. Selección de Columnas a Escalar

**Se EXCLUYEN del escalado:**
- `fecha_evento` y columnas de texto/ID (son categorías, no numéricas continuas)
- `month_sin`, `month_cos`, `trimestre_sin`, `trimestre_cos` (ya están acotadas entre −1 y 1, re-escalar distorsionaría su geometría circular)

**Se INCLUYEN todas las demás numéricas:** variables objetivo, rezagos, variables NASA y NLP.

In [ ]:
# Columnas a excluir del escalado
EXCLUDE_COLS = {
    'fecha_evento',
    # Cíclicas: ya están en [-1, 1]
    'month_sin', 'month_cos',
    'trimestre_sin', 'trimestre_cos',
    # IDs / texto
    'provincia', 'departamento', 'cultivo', 'region',
    'mes_num', 'trimestre_num',
}

# Detectar columnas numéricas continuas elegibles
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cols_to_scale = [c for c in numeric_cols if c not in EXCLUDE_COLS]

print(f"Total columnas numéricas : {len(numeric_cols)}")
print(f"Columnas excluidas       : {len([c for c in numeric_cols if c in EXCLUDE_COLS])}")
print(f"Columnas a escalar       : {len(cols_to_scale)}")
print(f"\nEjemplos de columnas a escalar: {cols_to_scale[:10]} ...")


## 3. División Temporal Train / Test (80/20)

Para evitar **Data Leakage**, el `MinMaxScaler` se ajusta (`fit`) **sólo** con los datos de entrenamiento (los primeros 80% cronológicos). Luego se aplica la transformación (`transform`) al dataset completo.

> ⚠️ En series de tiempo **NO** se hace un split aleatorio. Siempre se corta por fecha para respetar el orden causal: el pasado entrena, el futuro valida.

In [ ]:
# Ordenamos por fecha globalmente (ya viene ordenado, pero por seguridad)
df = df.sort_values('fecha_evento').reset_index(drop=True)

# Punto de corte: 80% cronológico
cutoff_idx  = int(len(df) * 0.80)
cutoff_date = df.loc[cutoff_idx, 'fecha_evento']

df_train = df.iloc[:cutoff_idx]
df_test  = df.iloc[cutoff_idx:]

print(f"Total muestras : {len(df):,}")
print(f"Train (80%)    : {len(df_train):,} → hasta {df_train['fecha_evento'].max().date()}")
print(f"Test  (20%)    : {len(df_test):,}  → desde {df_test['fecha_evento'].min().date()}")
print(f"\nFecha de corte: {cutoff_date.date()}")


## 4. Ajuste y Aplicación del MinMaxScaler

Ajustamos el scaler **sólo con Train** y luego lo aplicamos a todo el dataset. Guardamos el objeto scaler con `joblib` para poder invertir la transformación durante la inferencia futura (el modelo predice valores escalados; para interpretarlos debemos volver a la escala original).

In [ ]:
# Crear y ajustar el scaler SÓLO sobre Train
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(df_train[cols_to_scale])

# Transformar el dataset completo (Train + Test)
df_scaled = df.copy()
df_scaled[cols_to_scale] = scaler.transform(df[cols_to_scale])

print("✅ MinMaxScaler ajustado sobre Train y aplicado a todo el dataset.")
print(f"   Rango antes del escalado (produccion_t): "
      f"{df['produccion_t'].min():.2f} → {df['produccion_t'].max():.2f}"
      if 'produccion_t' in df.columns else "")
print(f"   Rango después del escalado (produccion_t): "
      f"{df_scaled['produccion_t'].min():.4f} → {df_scaled['produccion_t'].max():.4f}"
      if 'produccion_t' in df_scaled.columns else "")


In [ ]:
# ── Serializar el scaler principal ───────────────────────────────────────────
scaler_path = os.path.join(SCALERS_DIR, 'minmax_scaler_fase2.joblib')
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler guardado en: {os.path.abspath(scaler_path)}")

# ── Guardar también la lista de columnas escaladas ────────────────────────────
# (necesaria para reconstruir el scaler al hacer inferencia)
cols_path = os.path.join(SCALERS_DIR, 'cols_to_scale.joblib')
joblib.dump(cols_to_scale, cols_path)
print(f"✅ Lista de columnas guardada en: {os.path.abspath(cols_path)}")

# ── Verificación: cargar y comprobar ─────────────────────────────────────────
scaler_loaded = joblib.load(scaler_path)
print(f"\nVerificación: scaler recargado correctamente.")
print(f"  Tipo: {type(scaler_loaded)}")
print(f"  Feature range: {scaler_loaded.feature_range}")
print(f"  N° variables escaladas: {len(scaler_loaded.scale_)}")


## 5. Visualización: Distribución Antes y Después del Escalado

Confirmamos que las distribuciones de las variables principales quedaron correctamente acotadas entre 0 y 1.

In [ ]:
# Variables de interés para la visualización
VIZ_COLS = [c for c in ['produccion_t', 'T2M', 'PRECTOTCORR', 'sentiment_score']
            if c in df_scaled.columns]

if VIZ_COLS:
    fig, axes = plt.subplots(2, len(VIZ_COLS), figsize=(5 * len(VIZ_COLS), 8))
    if len(VIZ_COLS) == 1:
        axes = axes.reshape(2, 1)

    for i, col in enumerate(VIZ_COLS):
        # Antes del escalado
        axes[0, i].hist(df[col].dropna(), bins=40, color='steelblue', alpha=0.7)
        axes[0, i].set_title(f'{col}\nAntes del escalado', fontsize=10)
        axes[0, i].set_ylabel('Frecuencia')

        # Después del escalado
        axes[1, i].hist(df_scaled[col].dropna(), bins=40, color='darkorange', alpha=0.7)
        axes[1, i].set_title(f'{col}\nDespués del escalado [0, 1]', fontsize=10)
        axes[1, i].set_ylabel('Frecuencia')

    plt.suptitle('Distribución de Variables: Antes vs. Después del Escalado MinMax',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../../data/processed/visualizacion_escalado.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Gráfico guardado en data/processed/visualizacion_escalado.png")
else:
    print("No se encontraron columnas de ejemplo para visualizar.")


## 6. Verificación Final del Dataset

Comprobamos que el dataset resultante esté limpio, sin NaNs, y que las variables escaladas estén efectivamente entre 0 y 1.

In [ ]:
# Verificación de rangos
range_check = df_scaled[cols_to_scale].agg(['min', 'max'])
out_of_range = range_check.loc[:, (range_check.loc['min'] < -0.001) |
                                   (range_check.loc['max'] > 1.001)]

if out_of_range.empty:
    print("✅ Todas las variables escaladas están en el rango [0, 1].")
else:
    print(f"⚠️  Columnas fuera de rango (revisar):\n{out_of_range}")

# NaNs restantes
total_nans = df_scaled.isna().sum().sum()
print(f"\n{'✅' if total_nans == 0 else '⚠️ '} NaNs totales en el dataset: {total_nans}")
print(f"\nShape final del dataset: {df_scaled.shape[0]:,} filas × {df_scaled.shape[1]} columnas")
df_scaled.describe().round(3)


## 7. Exportar Dataset Final

Este es el **dataset maestro** que entrará directamente al modelo LSTM. Contiene:
- ✅ Sentimiento NLP (Act. 01)
- ✅ Codificación cíclica del tiempo y coordenadas (Act. 02)
- ✅ Rezagos temporales t-1, t-3, t-6, t-12 (Act. 03)
- ✅ Normalización MinMax [0, 1] (Act. 04)

**La Fase 2: Feature Engineering para LSTM está completa.**

In [ ]:
df_scaled.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Dataset final exportado a: {OUTPUT_FILE}")
print(f"   {df_scaled.shape[0]:,} filas × {df_scaled.shape[1]} columnas")
print(f"\nResumen de la Fase 2 completada:")
print(f"  📄 master_dataset_fase2_final.csv → dataset listo para LSTM")
print(f"  🔧 scalers/minmax_scaler_fase2.joblib → scaler para inferencia")
print(f"  🔧 scalers/cols_to_scale.joblib       → columnas del scaler")
